In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from metabolomics_toolkit.broadhurst import *
from metabolomics_toolkit.dimensionality_reduction import *
from metabolomics_toolkit.functions import *
import plotly.io as pio
pio.renderers.default = "notebook"
ROOT = "/users/j/a/jaileru/ST002118-Processing"

In [2]:
# Post-Processing
original = pd.read_csv(f"{ROOT}/results/ST002118_aligned_features.csv")
original = original.set_index("name")
original = original[original.columns[original.columns.str.contains("height")]]
original.columns = original.columns.str.replace(".mzML height","")
original = original.drop(columns=['height'])
original = original.fillna(original.min() * 0.10)
original = original.T
# original.to_parquet(f"{ROOT}/results/st002118_feature_table.parquet")
original.head()

name,#1 353.2658@16.0768 MS1+,#2 381.2971@16.5925 MS1+,#3 321.2396@15.5007 MS1+,#4 413.2659@16.7819 MS1+,#5 149.0232@13.8762 MS1+,#6 166.0862@3.2349 MS1+,#7 301.1408@13.8758 MS1+,#8 116.0706@0.7119 MS1+,#9 137.0458@1.2254 MS1+,#10 297.1671@11.5085 MS1+,...,#39322 133.9864@0.0670 MS1+,#39323 106.9353@2.2606 MS1+,#39324 194.9409@2.5464 MS1+,#39325 183.9324@0.3631 MS1+,#39326 273.8892@2.2908 MS1+,#39327 115.9394@0.0902 MS1+,#39328 73.0698@2.2774 MS1+,#39329 73.0598@2.4946 MS1+,#39330 149.9273@1.9970 MS1+,#39331 73.0762@0.1010 MS1+
Lich_NIST_1,6.874157e+09,3.739804e+09,6.519205e+09,5.038726e+09,6.109404e+09,5.370557e+09,4.806548e+09,2.587257e+09,3.106074e+08,5.911101e+05,...,43160.1050,2688.6072,2688.6072,2688.6072,2688.6072,2688.6072,2688.6072,2688.6072,2688.6072,40537.1130
Lich_S_1,7.372882e+09,7.827814e+09,3.156045e+07,2.355055e+08,2.650559e+07,3.834930e+09,1.052773e+08,1.862330e+09,2.917623e+09,2.622190e+09,...,30737.3340,2150.8039,2150.8039,2150.8039,29683.0370,2150.8039,2150.8039,2150.8039,2150.8039,2150.8039
Lich_S_2,7.109196e+09,8.410244e+09,5.192831e+06,2.655811e+08,9.353033e+06,3.495532e+09,3.184548e+07,2.878411e+09,2.749752e+09,2.082717e+09,...,2425.8387,2425.8387,2425.8387,46342.0590,39090.7540,2425.8387,2425.8387,36086.3440,2425.8387,32559.0200
Lich_S_3,5.893726e+09,4.515988e+09,3.485547e+06,2.603186e+08,5.592172e+06,3.603511e+09,1.785237e+07,2.191259e+09,1.510304e+09,1.521951e+09,...,2390.9543,2390.9543,2390.9543,2390.9543,2390.9543,2390.9543,2390.9543,2390.9543,36106.6640,2390.9543
Lich_S_4,7.052316e+09,6.191478e+09,2.076929e+06,3.032255e+08,2.840033e+06,4.055601e+09,9.309409e+06,2.688639e+09,5.039476e+08,3.169990e+09,...,1933.4906,1933.4906,1933.4906,1933.4906,1933.4906,1933.4906,1933.4906,31530.1330,1933.4906,34690.3870


In [3]:
metadata = pd.read_csv(f"{ROOT}/metadata/sample_metadata.csv").set_index("sample_name")[['batch','sample_type']]
metadata.head(5)

,batch,sample_type
sample_name,,
Lich_NIST_1,1,NIST Reference Material
Lich_S_1,1,Study Sample
Lich_S_2,1,Study Sample
Lich_S_3,1,Study Sample
Lich_S_4,1,Study Sample


In [17]:
pca_df = pca_plot(original,metadata,hue=['batch','sample_type'],backend='plotly',output_file="pre_correction.png",title='Pre-Correction')

In [5]:
# 2 - Submit Batch Effect Correction Pipeline (Normalization + Batch Effect Correction)
!cat ../scripts/st002118_batch_effect_correction.py

import sys
import os 
import pandas as pd 
import numpy as np
ROOT = "/users/j/a/jaileru/ST002118-Processing"
from serrf_py.base import Preprocessor,BatchCorrectionPipeline
from serrf_py.correlation_errors import SERRF

data = pd.read_parquet(f"{ROOT}/results/st002118_feature_table.parquet")
metadata = pd.read_csv(f"{ROOT}/metadata/sample_metadata.csv")
metadata = metadata.set_index("sample_name")
preprocessor = Preprocessor(
    imputation_method=None,
    normalization_method="TIC",
    transformation_method=None
)

corrector = SERRF(
    qc_str="_SP_",       # substring identifying QC samples in sample names
    blank_str="_B_", # substring identifying blank samples
    n_jobs=-1,         # parallel jobs for Random Forest fitting
    num_features=10,   # number of correlated features to use per signal
    use_ranger=True 
)
log_file = 'ST002118_11_16_2026.log' # specify log file or stdout will just be directed to the console
pipeline = BatchCorrectionPipeline(method=corrector, prepr

In [6]:
# 3 - Review Batch Effect and RSD Filtering Results
batch_corrected = pd.read_csv(f"{ROOT}/results/ST002118_Batch_Corrected.csv")
batch_corrected = batch_corrected.rename(columns={"Unnamed: 0":"name"}).set_index("name")

In [16]:
pca_df = pca_plot(batch_corrected,metadata,hue=['batch','sample_type'],backend='plotly',title='Batch Correction',output_file='post-correction.png')

In [10]:
# 4 - Apply RSD Filtering to remove disparate signals 
pca_df = pca_plot(CV(batch_corrected,qc='_NIST_',apply_filter=True,filter_by='nonparametric'),
                  metadata,hue=['batch','sample_type'],backend='plotly',title='RSD Filtered',applylog=True)